# Explore TabICLv2 regression and classification

Use Structured Data Models in a GPU Colab notebook to predict generated numerical targets and binary labels, then compare preprocessing, ensemble sizes, and cached inference.
Both datasets are synthetic; metrics demonstrate the API rather than external benchmark performance.
See [in-context learning](https://nvidia.github.io/structured-data-models/icl.html) for the model interface.


## Install

Install the package and the data/evaluation libraries in a GPU Colab runtime.


In [ ]:
%pip install structured-data-models scikit-learn pandas


## Set the example parameters

Choose the data seed, context size, estimator count, and output location. Eight estimators combine differently preprocessed views; larger ensembles take more compute.
The classification threshold converts class-1 probabilities to labels. Each run keeps data and predictions in a new local directory.


In [ ]:
import time
from pathlib import Path
from tempfile import mkdtemp
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sdm import TableTensor, infer_stypes
from sdm.models import TabICLv2

RANDOM_SEED = 42
ROW_COUNT = 7000
REGRESSION_CONTEXT_ROWS = 3000
CLASSIFICATION_TEST_FRACTION = 0.3
NUM_ESTIMATORS = 8
CLASSIFICATION_THRESHOLD = 0.5
OUTPUT_PARENT = Path.cwd()
device = torch.device("cuda")

run_dir = Path(mkdtemp(prefix="synthetic_tabicl-", dir=OUTPUT_PARENT))
print("Retained run:", run_dir)


## Generate synthetic example data

Run the next cell to create two independent tables: a noisy numerical target based on two features, and binary classes with different feature centers.
You can skim the generation code; model preparation starts below.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
features = rng.normal(size=(ROW_COUNT, 12))
regression = pd.DataFrame(features.round(4), columns=[f"feature_{i}" for i in range(12)])
regression["regression_target"] = (
    4 + features[:, 0] * 1.5 - features[:, 1] + rng.normal(0, 0.3, ROW_COUNT)
)
labels = rng.integers(0, 2, ROW_COUNT)
centers = rng.normal(size=(2, 12))
classification = pd.DataFrame(
    (centers[labels] + rng.normal(0, 0.5, (ROW_COUNT, 12))).round(4),
    columns=regression.columns[:-1],
)
classification["class_label"] = labels

data_dir = run_dir / "data"
data_dir.mkdir()
regression.to_csv(data_dir / "regression.csv", index=False)
classification.to_csv(data_dir / "classification.csv", index=False)
display(regression.head(3))
print("Rows per task:", ROW_COUNT)


## Prepare regression context

Convert the numerical table to a GPU `TableTensor`, reserve the first context rows, and keep held-out targets outside scoring features.
Load pretrained TabICLv2 weights; no task-specific training is needed.


In [ ]:
df = regression
stypes = infer_stypes(df)
table = TableTensor.from_pandas(df=df, stypes=stypes, device=device)
x = table.drop_columns("regression_target")
y = table[:, "regression_target"]
x_ctx, y_ctx = x[:REGRESSION_CONTEXT_ROWS], y[:REGRESSION_CONTEXT_ROWS]
x_qry = x[REGRESSION_CONTEXT_ROWS:]
y_true = df["regression_target"].to_numpy()[REGRESSION_CONTEXT_ROWS:]
query_ids = df.index.to_numpy()[REGRESSION_CONTEXT_ROWS:]
assert 1 < REGRESSION_CONTEXT_ROWS < len(df) - 1
print("Semantic types:", stypes)
print("Context rows:", len(x_ctx), "Scoring rows:", len(x_qry))
model = TabICLv2(device=device)


## Batch predictions

Predict every held-out regression row using labeled context. The prediction call receives only feature columns for the scoring population.


In [ ]:
with torch.inference_mode(), torch.amp.autocast(device.type, torch.bfloat16):
    out = model(
        x_context=x_ctx,
        y_context=y_ctx,
        x_query=x_qry,
        num_estimators=NUM_ESTIMATORS,
        generator=torch.Generator(device=device).manual_seed(RANDOM_SEED),
    )


### Inspect and save regression predictions

Expect one row per input. `row_id` identifies the generated row and `prediction` is the mean of the predicted quantiles, in the synthetic target's units.
Verify row alignment and finite values, then save the full batch.


In [ ]:
# Average the 999 predicted quantiles to obtain one estimate per row.
pred = out.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)
assert len(pred) == len(query_ids) and len(pred) >= 2
assert np.isfinite(pred).all()
batch_results = pd.DataFrame({"row_id": query_ids, "prediction": pred})
assert batch_results.row_id.tolist() == query_ids.tolist()
display(batch_results.head(8))
output_path = run_dir / "regression_predictions.csv"
batch_results.to_csv(output_path, index=False)
print("Saved:", output_path)


### Evaluate regression

**Synthetic-data metrics — demonstration only.** RMSE measures typical error in target units; lower is better. R² compares squared error with a constant mean prediction; higher is better.


In [ ]:
print("Synthetic-data metrics — demonstration only")
print(f"RMSE={mean_squared_error(y_true, pred) ** 0.5:.4f}  R2={r2_score(y_true, pred):.4f}")


## Prepare classification context

Use the same model interface for the independently generated binary task. Stratify the split to keep generated classes `0` and `1` in both populations.


In [ ]:
class_table = TableTensor.from_pandas(
    df=classification,
    stypes=infer_stypes(classification, overrides={"class_label": "categorical"}),
    device=device,
)
ctx_idx, qry_idx = train_test_split(
    np.arange(len(classification)),
    test_size=CLASSIFICATION_TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=classification["class_label"],
)
assert set(ctx_idx).isdisjoint(qry_idx)
ctx = torch.as_tensor(ctx_idx, device=device)
qry = torch.as_tensor(qry_idx, device=device)
class_features = class_table.drop_columns("class_label")
class_labels = class_table[:, "class_label"]
print("Context rows:", len(ctx_idx), "Scoring rows:", len(qry_idx))


### Predict the classification batch

Score the held-out features with labeled classification context; target values stay outside the query features.


In [ ]:
with torch.inference_mode(), torch.amp.autocast(device.type, torch.bfloat16):
    class_output = model(
        x_context=class_features[ctx],
        y_context=class_labels[ctx],
        x_query=class_features[qry],
        num_estimators=NUM_ESTIMATORS,
        generator=torch.Generator(device=device).manual_seed(RANDOM_SEED),
    )


### Inspect and save classification predictions

Expect one output row per input. The two probability columns correspond to generated classes `0` and `1`; `prediction` uses the configured class-1 threshold.
Check probability totals and row IDs before saving.


In [ ]:
p0 = class_output[:, "0"].numerical.float().cpu().numpy().reshape(-1)
p1 = class_output[:, "1"].numerical.float().cpu().numpy().reshape(-1)
assert len(p0) == len(p1) == len(qry_idx)
assert np.isfinite(np.column_stack([p0, p1])).all()
np.testing.assert_allclose(p0 + p1, 1, atol=1e-5)
class_predictions = (p1 > CLASSIFICATION_THRESHOLD).astype(int)
classification_results = pd.DataFrame({
    "row_id": qry_idx,
    "prediction": class_predictions,
    "probability_class_0": p0,
    "probability_class_1": p1,
})
assert classification_results.row_id.tolist() == qry_idx.tolist()
display(classification_results.head(8))
classification_results.to_csv(run_dir / "classification_predictions.csv", index=False)
print("Saved:", run_dir / "classification_predictions.csv")


### Evaluate classification

**Synthetic-data metrics — demonstration only.** Accuracy measures correct generated labels; ROC AUC measures ranking of class `1` above class `0`. Higher is better for both metrics.


In [ ]:
y_class = classification["class_label"].to_numpy()[qry_idx]
print("Synthetic-data metrics — demonstration only")
print(f"ROC AUC={roc_auc_score(y_class, p1):.4f}  accuracy={accuracy_score(y_class, class_predictions):.4f}")


## Compare preprocessing recipes

Optionally return to the regression task and compare a quantile transform with the default numerical preprocessing. Keep context, query rows, estimator count, and random seed fixed.
See [processing recipes](https://nvidia.github.io/structured-data-models/api/processing.html) for the available processors.


In [ ]:
default_recipe = model.default_recipe()
default_recipe


Define the alternative numerical recipe and a prediction helper for the remaining regression comparisons.


In [ ]:
from sdm.processing import (
    AverageEstimators, Clip, ClipSigma, DropConstantColumns, ImputeMean,
    QuantileTransform, Recipe, ShuffleColumns, Standardize,
)

custom_recipe = Recipe(
    features=[
        ImputeMean(),
        DropConstantColumns(),
        Standardize(eps=1e-6),
        Clip(min_value=-100.0, max_value=100.0),
        QuantileTransform(output_distribution="normal"),
        ClipSigma(threshold=4.0),
        ShuffleColumns(method="latin"),
    ],
    target=Standardize(),
    output=AverageEstimators(),
)

def predict_regression(recipe=None, estimators=NUM_ESTIMATORS):
    with torch.inference_mode(), torch.amp.autocast(device.type, torch.bfloat16):
        result = model(
            x_context=x_ctx, y_context=y_ctx, x_query=x_qry,
            num_estimators=estimators, recipe=recipe,
            generator=torch.Generator(device=device).manual_seed(RANDOM_SEED),
        )
    return result.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)


Compare errors on the same held-out regression rows. The result depends on this generated dataset; an improvement is not assumed.
**Synthetic-data metrics — demonstration only.**


In [ ]:
default_predictions = predict_regression()
quantile_predictions = predict_regression(recipe=custom_recipe)
print("Synthetic-data metrics — demonstration only")
print(f"Default RMSE: {mean_squared_error(y_true, default_predictions) ** 0.5:.4f}")
print(f"Quantile-transform RMSE: {mean_squared_error(y_true, quantile_predictions) ** 0.5:.4f}")


## Measure repeated inference with caching

Encode the regression context once with `fit()`, then reuse it with `predict()`. Compare average repeated-call latency after warmup; the cached timing excludes the one-time `fit()` cost.
Timing depends on the runtime and is separate from prediction quality.


In [ ]:
def timed(fn, warmup=1, iters=3):
    """Mean seconds per call, excluding one-time warmup costs."""
    for _ in range(warmup):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


# Caching disabled: re-encode the context on every call.
def predict():
    model(x_context=x_ctx, y_context=y_ctx, x_query=x_qry, num_estimators=NUM_ESTIMATORS)


# Caching enabled: encode + cache the context once, then reuse it per query.
def predict_with_cache():
    model.predict(x_qry)


with torch.inference_mode(), torch.amp.autocast(device.type, torch.bfloat16):
    t_nocache = timed(predict)
    model.fit(x=x_ctx, y=y_ctx, num_estimators=NUM_ESTIMATORS)
    t_cache = timed(predict_with_cache)
model.clear()

print(f"no cache: {t_nocache * 1e3:6.1f} ms")
print(f"cache:    {t_cache * 1e3:6.1f} ms")
print(f"speedup:  {t_nocache / t_cache:.1f}x")


## Compare ensemble sizes

Compare regression RMSE for 1, 8, and 32 estimators on the same rows. More estimators combine additional preprocessed views and increase compute; they need not improve this synthetic task.
**Synthetic-data metrics — demonstration only.**


In [ ]:
print("Synthetic-data metrics — demonstration only")
for estimators in [1, 8, 32]:
    ensemble_predictions = predict_regression(estimators=estimators)
    rmse = mean_squared_error(y_true, ensemble_predictions) ** 0.5
    print(f"Estimators={estimators:2d}  RMSE={rmse:.4f}")


## Continue with TabICLv2

Keep `run_dir` for generated data and both prediction CSVs. See [in-context learning](https://nvidia.github.io/structured-data-models/icl.html) for choosing between one-shot and cached prediction.
